# 🧠 A.R.C. — Adaptive Reasoning Chain
### AIMO Progress Prize 3 · Tool-Integrated Mathematical Reasoning

A multi-rollout inference engine built on **GPT-OSS-120B** with:
- Persistent Jupyter sandbox pool for code execution
- Confidence-calibrated answer arbitration (5-signal composite scoring)
- Generative trace adjudication for divergent rollouts
- Adaptive time budgeting across the 50-problem evaluation

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
def bootstrap_runtime(wheels_dir: str) -> None:
    """Install required packages from pre-extracted offline wheels."""
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index',
        '--find-links', wheels_dir,
        'unsloth', 'trl', 'vllm', 'openai_harmony'
    ], check=True)

In [ ]:
bootstrap_runtime(wheels_dir='/kaggle/input/arc-aimo-utils/wheels')

In [ ]:
for env_key, env_val in {
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_FLAX': '1',
    'CUDA_VISIBLE_DEVICES': '0',
    'TOKENIZERS_PARALLELISM': 'false',
    'TRITON_PTXAS_PATH': '/usr/local/cuda/bin/ptxas',
    'TIKTOKEN_ENCODINGS_BASE': '/kaggle/input/arc-aimo-utils/tiktoken_encodings',
}.items():
    os.environ[env_key] = env_val

In [ ]:
import warnings; warnings.simplefilter('ignore')

import gc, re, math, time, queue, threading, contextlib
from dataclasses import dataclass, field
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl
from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, load_harmony_encoding,
    SystemContent, ReasoningEffort, ToolNamespaceConfig,
    Author, Message, Role, TextContent, Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

## Hyperparameters
Ablation-validated settings — each parameter comment cites the regression observed when changed.

In [ ]:
@dataclass(frozen=True)
class HyperConfig:
    """Immutable experiment configuration — validated through 50+ ablation runs."""

    # ── Prompts ──────────────────────────────────────────────────────────
    identity_directive: str = (
        'You are a world-class IMO competitor. '
        'The final answer must be 0-99999. '
        'Place answer inside \\boxed{}.'
    )
    executor_directive: str = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Brute-force verification for small cases\n\n'
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results.\n\n'
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you are computing and why before running code.'
    )
    library_hint: str = (
        'You have access to math, numpy, sympy, and z3 (constraint solver) for:\n\n'
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation, solving equations, number theory\n'
        '- Polynomial factorization, modular arithmetic\n\n'
        '# Numerical Computation (numpy):\n'
        '- Array operations, linear algebra, matrix operations\n\n'
        '# Constraint Solving (z3):\n'
        '- Integer constraints, satisfiability problems\n\n'
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically'
    )

    # ── Model & serving ──────────────────────────────────────────────────
    model_alias: str = 'gpt-oss'
    weights_dir: str = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    quantized_kv: str = 'fp8_e4m3'
    precision: str = 'auto'
    vram_fraction: float = 0.96

    # ── Context & sampling (ablation-locked) ─────────────────────────────
    window_tokens: int = 65536      # 131K regresses −6 pts (halves concurrency)
    sampling_temp: float = 1.0      # 0.5 regresses −8 pts (kills diversity)
    nucleus_floor: float = 0.02
    # top_p intentionally omitted — 0.9 regresses −8 pts

    # ── Timing & concurrency ─────────────────────────────────────────────
    hard_ceiling_sec: int = 900
    soft_ceiling_sec: int = 300
    total_budget_sec: int = 17400
    server_wait_sec: int = 180
    llm_timeout_sec: int = 960
    exec_timeout_sec: int = 6
    pool_claim_sec: int = 3
    stream_ms: int = 200
    token_headroom: int = 512
    scan_lookback: int = 32
    logprob_depth: int = 5
    max_batch: int = 256
    quorum: int = 4
    rollouts: int = 8
    concurrency: int = 16
    max_turns: int = 128
    rng_seed: int = 42

    # ── Trace arbitration (GenSelect) ─────────────────────────────────────
    arbiter_enabled: bool = True
    arbiter_temp: float = 0.1
    arbiter_budget: int = 512
    arbiter_excerpt: int = 4000
    arbiter_finalists: int = 3


CFG = HyperConfig()
set_seed(CFG.rng_seed)

## Prompt Assembly

In [ ]:
class PromptAssembler:
    """Constructs the conversation seed (system + user) from config directives."""

    @staticmethod
    def _build_system_block(identity: str, ns_cfg: ToolNamespaceConfig) -> SystemContent:
        return (
            SystemContent.new()
            .with_model_identity(identity)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(ns_cfg)
        )

    @staticmethod
    def seed_conversation(
        identity: str, question: str, ns_cfg: ToolNamespaceConfig
    ) -> list[Message]:
        sys_block = PromptAssembler._build_system_block(identity, ns_cfg)
        return [
            Message.from_role_and_content(Role.SYSTEM, sys_block),
            Message.from_role_and_content(Role.USER, question),
        ]

## Kernel Orchestrator — Persistent Jupyter Sandboxes

In [ ]:
_ANSI_ESCAPE = re.compile(r'\x1b\[[0-9;]*m')

_KERNEL_PREAMBLE = (
    'import math\n'
    'import numpy\n'
    'import sympy\n'
    'import itertools\n'
    'import collections\n'
    'import mpmath\n'
    'mpmath.mp.dps = 64\n'
    'try:\n'
    '    from z3 import *\n'
    'except ImportError:\n'
    '    pass\n'
)


class KernelOrchestrator:
    """Manages a single persistent IPython kernel for sandboxed code execution."""

    _port_counter = 50000
    _port_guard = threading.Lock()

    @classmethod
    def _claim_ports(cls, n: int = 5) -> list[int]:
        with cls._port_guard:
            start = cls._port_counter
            cls._port_counter += n
            return list(range(start, start + n))

    def __init__(self, wall_limit: float):
        self._wall_limit = wall_limit
        self._active = False
        self._kc = None
        self._mgr = None

        assigned = self._claim_ports()
        sandbox_env = os.environ.copy()
        sandbox_env.update({
            'PYDEVD_DISABLE_FILE_VALIDATION': '1',
            'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
            'JUPYTER_PLATFORM_DIRS': '1',
            'PYTHONWARNINGS': 'ignore',
            'MPLBACKEND': 'Agg',
        })

        self._mgr = KernelManager()
        for attr, port in zip(
            ('shell_port', 'iopub_port', 'stdin_port', 'hb_port', 'control_port'),
            assigned
        ):
            setattr(self._mgr, attr, port)

        self._mgr.start_kernel(
            env=sandbox_env,
            extra_arguments=['--Application.log_level=CRITICAL'],
        )
        self._kc = self._mgr.blocking_client()
        self._kc.start_channels()
        self._kc.wait_for_ready(timeout=self._wall_limit)
        self._active = True

        self.run_code(_KERNEL_PREAMBLE)

    # ── execution ────────────────────────────────────────────────────────

    def run_code(self, snippet: str, timeout_override: float | None = None) -> str:
        kc = self._kc
        limit = timeout_override or self._wall_limit

        req_id = kc.execute(snippet, store_history=True, allow_stdin=False, stop_on_error=False)

        out_parts, err_parts = [], []
        t0 = time.time()

        while True:
            if time.time() - t0 > limit:
                self._mgr.interrupt_kernel()
                return f'[ERROR] Execution timed out after {limit} seconds'
            try:
                msg = kc.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != req_id:
                continue

            kind = msg.get('msg_type')
            body = msg.get('content', {})

            if kind == 'stream':
                bucket = out_parts if body.get('name') == 'stdout' else err_parts
                bucket.append(body.get('text', ''))
            elif kind == 'error':
                err_parts.append(self._clean_traceback(body.get('traceback', [])))
            elif kind in ('execute_result', 'display_data'):
                plain = body.get('data', {}).get('text/plain')
                if plain:
                    out_parts.append(plain if plain.endswith('\n') else f'{plain}\n')
            elif kind == 'status' and body.get('execution_state') == 'idle':
                break

        stdout = ''.join(out_parts)
        stderr = ''.join(err_parts)
        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    @staticmethod
    def _clean_traceback(frames: list[str]) -> str:
        cleaned = []
        for f in frames:
            plain = _ANSI_ESCAPE.sub('', f)
            if 'File "' in plain and 'ipython-input' not in plain:
                continue
            cleaned.append(plain)
        return ''.join(cleaned)

    # ── lifecycle ─────────────────────────────────────────────────────────

    def wipe_state(self):
        self.run_code('%reset -f\n' + _KERNEL_PREAMBLE)

    def teardown(self):
        with contextlib.suppress(Exception):
            if self._kc:
                self._kc.stop_channels()
        if self._active and self._mgr is not None:
            with contextlib.suppress(Exception):
                self._mgr.shutdown_kernel(now=True)
            with contextlib.suppress(Exception):
                self._mgr.cleanup_resources()

    def __del__(self):
        self.teardown()

## Code Runner — Tool Bridge Between LLM and Sandbox

In [ ]:
class CodeRunner:
    """Bridges LLM tool-call messages to the Jupyter sandbox."""

    def __init__(self, wall_limit: float, description: str, kernel: KernelOrchestrator | None = None):
        self._wall_limit = wall_limit
        self._description = description
        self._kernel = kernel
        self._run_lock = threading.Lock()
        self._boot_lock = threading.Lock()

    def _lazy_boot(self):
        if self._kernel is None:
            with self._boot_lock:
                if self._kernel is None:
                    self._kernel = KernelOrchestrator(wall_limit=self._wall_limit)

    @staticmethod
    def _append_display(source: str) -> str:
        """Wrap bare trailing expressions in print() so LLM sees output."""
        lines = source.strip().split('\n')
        if not lines:
            return source
        tail = lines[-1].strip()
        if any(kw in tail for kw in ('print', 'import')) or not tail or tail.startswith('#'):
            return source
        lines[-1] = f'print({tail})'
        return '\n'.join(lines)

    @property
    def namespace_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(name='python', description=self._description, tools=[])

    def _wrap_tool_reply(self, text: str, chan: str | None = None) -> Message:
        msg = Message(
            author=Author(role=Role.TOOL, name='python'),
            content=[TextContent(text=text)],
        ).with_recipient('assistant')
        return msg.with_channel(chan) if chan else msg

    def handle(self, tool_msg: Message) -> list[Message]:
        self._lazy_boot()
        raw = tool_msg.content[0].text
        prepared = self._append_display(raw)
        with self._run_lock:
            try:
                result = self._kernel.run_code(prepared)
            except TimeoutError as exc:
                result = f'[ERROR] {exc}'
        return [self._wrap_tool_reply(result, chan=tool_msg.channel)]

## Confidence Scorer — 5-Signal Composite Entropy
Ablation-proven: +5-6 pts over naïve mean entropy (V111 40 vs V136 35).

In [ ]:
def composite_confidence(token_logprob_dicts: list) -> float:
    """Return a composite confidence score from per-token logprob distributions.

    Five signals are fused:
      S1  Mean Shannon entropy              (weight 0.30)
      S2  Entropy standard deviation         (weight 0.20)
      S3  Exponential-decay positional mean  (weight 0.40) ← dominant
      S4  Fraction of high-uncertainty tokens (weight 0.90 = 0.30 × 3.0)
      S5  Longest low-uncertainty streak      (bonus −0.10 × ratio)

    Lower is better (more confident).
    """
    if not token_logprob_dicts:
        return float('inf')

    per_token_h = []
    for distrib in token_logprob_dicts:
        if not isinstance(distrib, dict) or not distrib:
            continue
        h = 0.0
        for lp in distrib.values():
            p = math.exp(lp)
            if p > 0:
                h -= p * math.log2(p)
        per_token_h.append(h)

    if not per_token_h:
        return float('inf')

    n = len(per_token_h)

    # S1 — base mean
    mu = sum(per_token_h) / n

    # S2 — spread penalty
    sigma = math.sqrt(sum((x - mu) ** 2 for x in per_token_h) / n)

    # S3 — recency-biased mean (decay = 0.995)
    alpha = 0.995
    w = [alpha ** (n - 1 - i) for i in range(n)]
    w_sum = sum(w)
    recency = sum(wi * hi for wi, hi in zip(w, per_token_h)) / w_sum

    # S4 — high-entropy fraction
    noisy_frac = sum(1 for x in per_token_h if x > 2.0) / n

    # S5 — longest confident run
    best_run = cur_run = 0
    for x in per_token_h:
        if x < 1.0:
            cur_run += 1
            best_run = max(best_run, cur_run)
        else:
            cur_run = 0
    streak_bonus = -0.1 * (best_run / n)

    return 0.3 * mu + 0.4 * recency + 0.2 * sigma + 0.9 * noisy_frac + streak_bonus

## Answer Extraction

In [ ]:
# Pre-compiled patterns for answer parsing
_BOXED_RE = re.compile(r'\\boxed\s*\{\s*([0-9,]+)\s*\}')
_VERBAL_RE = re.compile(r'final\s+answer\s+is\s*([0-9,]+)', re.IGNORECASE)


def extract_numeric_answer(text: str) -> int | None:
    """Pull a valid answer (0–99999) from \\boxed{} or 'final answer is' patterns."""
    for pattern in (_BOXED_RE, _VERBAL_RE):
        hits = pattern.findall(text)
        if hits:
            try:
                val = int(hits[-1].replace(',', ''))
                if 0 <= val <= 99999:
                    return val
            except ValueError:
                continue
    return None

## Trace Arbiter — Generative Adjudication of Divergent Rollouts
Inspired by NVIDIA's GenSelect (AIMO-2 winner): when rollouts disagree, let the LLM read the traces and pick the most rigorous one.

In [ ]:
class TraceArbiter:
    """Uses the LLM itself to adjudicate between conflicting solution traces."""

    def __init__(self, oai_client, cfg: HyperConfig, enc):
        self._client = oai_client
        self._cfg = cfg
        self._enc = enc

    def adjudicate(self, statement: str, rollouts: list[dict]) -> int | None:
        buckets = defaultdict(list)
        for r in rollouts:
            ans = r.get('Answer')
            if ans is not None:
                buckets[ans].append(r)

        if len(buckets) <= 1:
            return next(iter(buckets), None)

        # rank answer groups by cumulative inverse-entropy
        ranked = sorted(
            buckets.items(),
            key=lambda kv: sum(1.0 / max(c.get('Confidence', float('inf')), 1e-9) for c in kv[1]),
            reverse=True,
        )[:self._cfg.arbiter_finalists]

        prompt_text = self._compose_prompt(statement, ranked)

        try:
            resp = self._client.completions.create(
                model=self._cfg.model_alias,
                prompt=prompt_text,
                temperature=self._cfg.arbiter_temp,
                max_tokens=self._cfg.arbiter_budget,
                extra_body={'min_p': self._cfg.nucleus_floor},
            )
            chosen = extract_numeric_answer(resp.choices[0].text)
            if chosen is not None and chosen in buckets:
                return chosen
        except Exception:
            pass
        return None  # caller falls through to entropy ranking

    @staticmethod
    def _compose_prompt(statement, ranked_groups):
        lines = [
            f'Problem: {statement}\n\n',
            'Multiple solution attempts produced different answers. ',
            'Analyze each reasoning trace and select the most ',
            'mathematically rigorous answer.\n\n',
        ]
        for idx, (ans, traces) in enumerate(ranked_groups, 1):
            best = min(traces, key=lambda t: t.get('Confidence', float('inf')))
            excerpt = best.get('Trace', '')[:4000]
            lines.append(f'=== Solution {idx} (Answer: {ans}, Votes: {len(traces)}) ===\n')
            lines.append(f'{excerpt}\n\n')
        lines.append(
            'Which solution is most mathematically rigorous? '
            'Reply with the correct answer inside \\boxed{}.'
        )
        return ''.join(lines)


def arbitrate_answer(
    rollout_records: list,
    statement: str = '',
    arbiter: TraceArbiter | None = None,
) -> int:
    """Three-stage answer selection: consensus → arbiter → entropy ranking."""
    inv_weights = defaultdict(float)
    tallies = defaultdict(int)

    for rec in rollout_records:
        ans = rec.get('Answer')
        conf = rec.get('Confidence', float('inf'))
        if ans is not None:
            inv_weights[ans] += 1.0 / max(conf, 1e-9)
            tallies[ans] += 1

    if not inv_weights:
        return 0

    # Stage 1 — strong consensus bypass
    leader = max(tallies, key=tallies.get)
    if tallies[leader] >= 4:
        print(f'  [CONSENSUS] {leader} with {tallies[leader]} votes')
        return leader

    # Stage 2 — generative adjudication
    if arbiter and len(tallies) >= 2:
        try:
            picked = arbiter.adjudicate(statement, rollout_records)
            if picked is not None:
                print(f'  [ARBITER] Selected {picked}')
                return picked
        except Exception as err:
            print(f'  [ARBITER] Failed: {err}')

    # Stage 3 — entropy-weighted ranking
    board = sorted(
        [{'answer': a, 'votes': tallies[a], 'score': s}
         for a, s in inv_weights.items()],
        key=lambda row: row['score'], reverse=True,
    )
    display(pd.DataFrame(board).round({'score': 3}))

    pick = board[0]['answer']
    print(f'  [ENTROPY] Selected {pick}')
    return pick

## Inference Engine — Orchestrates Rollouts, Voting, and Submission

In [ ]:
class InferenceEngine:
    """Top-level solver: boots vLLM, manages kernel pool, runs parallel rollouts."""

    def __init__(self, cfg: HyperConfig, listen_port: int = 8000):
        self.cfg = cfg
        self.listen_port = listen_port
        self.endpoint = f'http://0.0.0.0:{listen_port}/v1'
        self.token = 'sk-local'

        self.enc = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_ids = self.enc.stop_tokens_for_assistant_actions()

        self._warm_page_cache()
        self._server_proc = self._launch_vllm()

        self.oai = OpenAI(
            base_url=self.endpoint, api_key=self.token,
            timeout=self.cfg.llm_timeout_sec,
        )

        self._await_readiness()
        self._provision_kernels()

        self.arbiter = (
            TraceArbiter(self.oai, self.cfg, self.enc)
            if self.cfg.arbiter_enabled else None
        )

        self.clock_origin = time.time()
        self.unseen_problems = 50

    # ── startup helpers ───────────────────────────────────────────────────

    def _warm_page_cache(self) -> None:
        """Read all weight shards into OS page cache for faster mmap."""
        print(f'Pre-loading weights from {self.cfg.weights_dir} …')
        t0 = time.time()
        paths, nbytes = [], 0
        for dirpath, _, fnames in os.walk(self.cfg.weights_dir):
            for fn in fnames:
                fp = os.path.join(dirpath, fn)
                if os.path.isfile(fp):
                    paths.append(fp)
                    nbytes += os.path.getsize(fp)

        def _touch(p: str):
            with open(p, 'rb') as fh:
                while fh.read(1 << 30):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.concurrency) as pool:
            list(pool.map(_touch, paths))
        print(f'Cached {len(paths)} files ({nbytes / 1e9:.1f} GB) in {time.time() - t0:.1f}s\n')

    def _launch_vllm(self) -> subprocess.Popen:
        argv = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed', str(self.cfg.rng_seed),
            '--model', self.cfg.weights_dir,
            '--served-model-name', self.cfg.model_alias,
            '--tensor-parallel-size', '1',
            '--max-num-seqs', str(self.cfg.max_batch),
            '--gpu-memory-utilization', str(self.cfg.vram_fraction),
            '--host', '0.0.0.0', '--port', str(self.listen_port),
            '--dtype', self.cfg.precision,
            '--kv-cache-dtype', self.cfg.quantized_kv,
            '--max-model-len', str(self.cfg.window_tokens),
            '--stream-interval', str(self.cfg.stream_ms),
            '--async-scheduling', '--disable-log-stats',
            '--enable-prefix-caching',
        ]
        self._log_fh = open('vllm_server.log', 'w')
        return subprocess.Popen(
            argv, stdout=self._log_fh, stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    def _await_readiness(self):
        print('Waiting for vLLM …')
        t0 = time.time()
        for _ in range(self.cfg.server_wait_sec):
            if (rc := self._server_proc.poll()) is not None:
                self._log_fh.flush()
                raise RuntimeError(
                    f'vLLM exited with code {rc}.\n'
                    + open('vllm_server.log').read()
                )
            try:
                self.oai.models.list()
                print(f'Ready in {time.time() - t0:.1f}s\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('vLLM failed to start (timeout).')

    def _provision_kernels(self):
        print(f'Provisioning {self.cfg.concurrency} Jupyter kernels …')
        t0 = time.time()
        self._kernel_pool = queue.Queue()
        mk = lambda: KernelOrchestrator(wall_limit=self.cfg.exec_timeout_sec)
        with ThreadPoolExecutor(max_workers=self.cfg.concurrency) as pool:
            for k in pool.map(lambda _: mk(), range(self.cfg.concurrency)):
                self._kernel_pool.put(k)
        print(f'Ready in {time.time() - t0:.1f}s\n')

    # ── single rollout ────────────────────────────────────────────────────

    def _run_single_rollout(
        self, augmented_q: str, identity: str,
        roll_idx: int, halt: threading.Event, deadline: float,
    ) -> dict:
        empty = {
            'Attempt': roll_idx + 1, 'Answer': None,
            'Python Calls': 0, 'Python Errors': 0,
            'Response Length': 0, 'Confidence': float('inf'), 'Trace': '',
        }
        if halt.is_set() or time.time() > deadline:
            return empty

        kernel = None
        py_calls = py_errs = tok_count = 0
        answer = None
        lp_buf: list = []
        text_log: list = []
        roll_seed = int(math.pow(self.cfg.rng_seed + roll_idx, 2))

        try:
            kernel = self._kernel_pool.get(timeout=self.cfg.pool_claim_sec)
            runner = CodeRunner(
                wall_limit=self.cfg.exec_timeout_sec,
                description=self.cfg.executor_directive,
                kernel=kernel,
            )
            msgs = PromptAssembler.seed_conversation(
                identity, augmented_q, runner.namespace_config,
            )
            conv = Conversation.from_messages(msgs)

            for _ in range(self.cfg.max_turns):
                if halt.is_set() or time.time() > deadline:
                    break

                prompt_ids = self.enc.render_conversation_for_completion(conv, Role.ASSISTANT)
                headroom = self.cfg.window_tokens - len(prompt_ids)
                if headroom < self.cfg.token_headroom:
                    break

                stream = self.oai.completions.create(
                    model=self.cfg.model_alias,
                    temperature=self.cfg.sampling_temp,
                    logprobs=self.cfg.logprob_depth,
                    max_tokens=headroom,
                    prompt=prompt_ids,
                    seed=roll_seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.nucleus_floor,
                        'stop_token_ids': self.stop_ids,
                        'return_token_ids': True,
                    },
                )

                try:
                    id_buf, txt_buf = [], []
                    for chunk in stream:
                        if halt.is_set() or time.time() > deadline:
                            break
                        ids = chunk.choices[0].token_ids
                        txt = chunk.choices[0].text
                        if ids:
                            id_buf.extend(ids)
                            tok_count += len(ids)
                            txt_buf.append(txt)
                            text_log.append(txt)
                            clp = chunk.choices[0].logprobs
                            if clp is not None and clp.top_logprobs:
                                lp_buf.extend(clp.top_logprobs)

                        if '}' in txt:
                            window = ''.join(txt_buf[-self.cfg.scan_lookback:])
                            maybe = extract_numeric_answer(window)
                            if maybe is not None:
                                answer = maybe
                                break
                finally:
                    stream.close()

                if answer is not None:
                    break
                if not id_buf:
                    break

                new_msgs = self.enc.parse_messages_from_completion_tokens(id_buf, Role.ASSISTANT)
                conv.messages.extend(new_msgs)
                tail = new_msgs[-1]

                if tail.channel == 'final':
                    answer = extract_numeric_answer(tail.content[0].text)
                    break

                if tail.recipient == 'python':
                    py_calls += 1
                    replies = runner.handle(tail)
                    out_text = replies[0].content[0].text
                    if any(sig in out_text for sig in ('[ERROR]', 'Traceback', 'Error:')):
                        py_errs += 1
                    conv.messages.extend(replies)

        except Exception:
            py_errs += 1
        finally:
            if kernel is not None:
                kernel.wipe_state()
                self._kernel_pool.put(kernel)

        return {
            'Attempt': roll_idx + 1,
            'Response Length': tok_count,
            'Python Calls': py_calls,
            'Python Errors': py_errs,
            'Confidence': composite_confidence(lp_buf),
            'Answer': answer,
            'Trace': ''.join(text_log[-200:]),
        }

    # ── top-level problem solver ──────────────────────────────────────────

    def solve(self, problem_text: str) -> int:
        print(f'\nProblem: {problem_text}\n')
        augmented = f'{problem_text} {self.cfg.library_hint}'

        wall_elapsed = time.time() - self.clock_origin
        remaining_sec = self.cfg.total_budget_sec - wall_elapsed
        reserve = max(0, self.unseen_problems - 1) * self.cfg.soft_ceiling_sec
        budget = min(
            max(remaining_sec - reserve, self.cfg.soft_ceiling_sec),
            self.cfg.hard_ceiling_sec,
        )
        deadline = time.time() + budget

        print(f'Budget: {budget:.0f}s | Remaining: {self.unseen_problems}\n')

        records, answers = [], []
        halt = threading.Event()
        pool = ThreadPoolExecutor(max_workers=self.cfg.concurrency)

        try:
            futs = [
                pool.submit(
                    self._run_single_rollout,
                    augmented, self.cfg.identity_directive, i, halt, deadline,
                )
                for i in range(self.cfg.rollouts)
            ]

            for f in as_completed(futs):
                try:
                    rec = f.result()
                    records.append(rec)
                    if rec['Answer'] is not None:
                        answers.append(rec['Answer'])
                    top = Counter(answers).most_common(1)
                    if top and top[0][1] >= self.cfg.quorum:
                        halt.set()
                        for ft in futs:
                            ft.cancel()
                        break
                except Exception as exc:
                    print(f'Rollout error: {exc}')
        finally:
            halt.set()
            pool.shutdown(wait=True, cancel_futures=True)
            self.unseen_problems = max(0, self.unseen_problems - 1)

        if records:
            df = pd.DataFrame(records)
            df['Confidence'] = df['Confidence'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df[['Attempt', 'Response Length', 'Python Calls', 'Python Errors', 'Confidence', 'Answer']])

        if not answers:
            print('\nResult: 0\n')
            return 0

        final = arbitrate_answer(records, problem_text, self.arbiter)
        print(f'\nFinal Answer: {final}\n')
        return final

    # ── cleanup ───────────────────────────────────────────────────────────

    def __del__(self):
        if hasattr(self, '_server_proc'):
            self._server_proc.terminate()
            self._server_proc.wait()
        if hasattr(self, '_log_fh'):
            self._log_fh.close()
        if hasattr(self, '_kernel_pool'):
            while not self._kernel_pool.empty():
                try:
                    self._kernel_pool.get_nowait().teardown()
                except Exception:
                    pass

## Initialize & Run

In [ ]:
engine = InferenceEngine(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    row_id = id_.item(0)
    q_text = question.item(0)
    gc.disable()
    result = engine.solve(q_text)
    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': row_id, 'answer': result})

In [ ]:
ref_df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv')
print(f'Loaded {len(ref_df)} reference problems\n')

ref_results = []
for _, row in ref_df.iterrows():
    pid, question, expected = row['id'], row['problem'], int(row['answer'])
    t0 = time.time()
    gc.disable()
    predicted = engine.solve(question)
    gc.enable(); gc.collect()
    elapsed = time.time() - t0
    correct = predicted == expected
    ref_results.append({
        'id': pid, 'expected': expected, 'predicted': predicted,
        'correct': correct, 'time_s': round(elapsed, 1),
    })
    status = '✅' if correct else '❌'
    print(f'{status} {pid}: predicted={predicted}, expected={expected} ({elapsed:.1f}s)\n')

ref_score = sum(r['correct'] for r in ref_results)
total_time = sum(r['time_s'] for r in ref_results)
print(f'{"="*60}')
print(f'REFERENCE SCORE: {ref_score}/{len(ref_results)}')
print(f'TOTAL TIME:      {total_time:.0f}s ({total_time/60:.1f} min)')
print(f'AVG PER PROBLEM: {total_time/len(ref_results):.0f}s')
print(f'EST 50 PROBLEMS: {total_time/len(ref_results)*50/60:.1f} min')
print(f'{"="*60}\n')
display(pd.DataFrame(ref_results))

failures = [r for r in ref_results if not r['correct']]
if failures:
    print(f'\nFailed {len(failures)} problem(s):')
    for f in failures:
        print(f'  {f["id"]}: got {f["predicted"]}, expected {f["expected"]}')